# Sorting with Multi-Head Attention

A transformer that learns to sort a sequence of digits. The model uses
learned embeddings, sinusoidal positional encoding, multi-head causal
self-attention, and layer normalization (Pre-LN architecture).

**Task:** Given `[3,1,4,0,2]`, predict `[0,1,2,3,4]` (with separator/EOS tokens).

**Reference:** Vaswani et al., "Attention Is All You Need" (2017)

**CLI equivalent:** `make example-transformer` (1000 epochs, batch=16)


## Architecture

The transformer maps a sequence of token indices to logits over the vocabulary.
All dimensions are checked at compile time.


In [ ]:
:browse Nn.Transformer

In [ ]:
:t TransformerBlock

The type parameters encode the full architecture:
- `seqLen`: maximum sequence length
- `dModel`: embedding dimension
- `numHeads`: number of attention heads
- `headDim`: dimension per head
- `numBlocks`: number of transformer blocks
- `vocabSize`: vocabulary size

Input dimension is `seqLen` (token indices), output is `seqLen * vocabSize` (logits).

### Sequence Format

For sorting 5 digits from vocab {0..5}:
```
Input:  [3, 1, 4, 0, 2, SEP, 0, 1, 2, 3, 4]  (teacher-forced)
Target: predict next token at each position
```
SEP=6, EOS=7, so vocab size is 8. Loss is only computed on the sorting
portion (after SEP).


## Training

We use a small config (dModel=32, 4 heads, 2 blocks) and train for 500 epochs
with AdamW. Fresh batches of random sorting problems are generated each epoch.

This cell takes ~15-20 seconds.


In [ ]:
:t attention

> **Note:** The cell above may produce a type constraint error in the REPL due to Idris 2's Peano Nat reduction limits at large dimensions. The CLI example (`make example-transformer`) compiles and runs correctly. See `docs/develop/gotchas.md` for details.


## Key Components

### Multi-Head Attention
Each head computes `softmax(QK^T / sqrt(d_k)) V` independently, then
results are summed (not concatenated, to keep output dimension = dModel).

### Causal Mask
The attention mask prevents positions from attending to future tokens,
enabling autoregressive generation.

### Pre-LN vs Post-LN
Layer normalization is applied *before* attention and FFN (Pre-LN),
which is more stable than the original Post-LN architecture.


## PyTorch Comparison

```python
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads):
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, 4*d_model),
                                nn.GELU(), nn.Linear(4*d_model, d_model))
```

In idris-ml, `mkTransformer` constructs the full stack. The type system
ensures `numHeads * headDim` is consistent with `dModel`, and the output
shape `seqLen * vocabSize` is enforced at compile time.

See `pytorch/torch_ref/scripts/transformer.py` for the full reference.


Next: [GPT](gpt.ipynb) — character-level language model using the same transformer.
